# 21 — Deterministic checks first, Ragas as a second signal
Ragas judges factual overlap with a local Ollama model. It cannot prove that
an approval gate or a customer scope was enforced.



## Problem

Contract tests cannot measure whether a recommendation is factually close to a reference.


## Concept

Ragas adds a model-judged score alongside deterministic safety gates.

Ragas FactualCorrectness calls an evaluator model to compare a response with a reference.


## Abstraction

reference + response → local Ollama evaluator → factual score. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
report → deterministic PASS/FAIL; response + reference → Ragas/Ollama score
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Run a deterministic report check

Run a deterministic report check.


In [ ]:
import asyncio
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.control import validate_report
from workshop.ragas_eval import factual_score, load_cases
from workshop.schemas import DecisionReport, Evidence

report = DecisionReport(
    recommendation="pilot",
    rationale="A limited pilot is justified while deletion guarantees remain unverified.",
    evidence=[Evidence(source_id="architecture.md", claim="Private VPC deployment is possible."),
              Evidence(source_id="security.md", claim="Deletion guarantees are unverified.")],
)
print("Deterministic report check:", validate_report(report))
cases = load_cases()
print("Versioned evaluation cases:", [case["id"] for case in cases])



## Run a local model-judged score only when requested
Scores may vary by model and run; record the model and input dataset.



## Step 2 — Compare response and reference with the local Ragas evaluator

Compare response and reference with the local Ragas evaluator.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    for case in cases:
        print(case["id"], asyncio.run(factual_score(case["response"], case["reference"])))
else:
    print("Set WORKSHOP_RUN_LIVE_DEMOS=true for the Ragas/Ollama metric.")


## Execution flow

reference + response → local Ollama evaluator → factual score. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

A high factual score cannot catch a missing approval or cross-customer memory leak.


## Production implication

Version the evaluation set and compare scores across model changes; never use one score alone.


## Your turn

Change the reference and compare the score while keeping safety tests fixed.

**Checkpoint:** The score is reported separately from validation and approval results.
